In [1]:
from dotenv import load_dotenv
import os 
import ast
import json
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

load_dotenv()

# Load base path
base_path = os.environ.get("BASE")
output_path = os.environ.get("OUTPUT")
input_path = os.environ.get("MAIN")

# Interractional Trouble

In [2]:
import pandas as pd 

# load base data
dataset = pd.read_csv(f'{input_path}/merged_turns.csv')
# Load tasks
turns_long = pd.read_csv(f'{input_path}/merged_tasks.csv')

In [3]:
# load trouble datasets 
grok_trouble   = pd.read_json(f'{output_path}grok/trouble.jsonl', lines=True)
claude_trouble = pd.read_json(f'{output_path}claude/trouble.jsonl', lines=True)
gemini_trouble = pd.read_json(f'{output_path}gemini/trouble.jsonl', lines=True)
chatgpt_trouble = pd.read_json(f'{output_path}chatgpt/trouble.jsonl', lines=True)
deepseek_trouble = pd.read_json(f'{output_path}deepseek/trouble.jsonl', lines=True)

# Interactional trouble across five platforms

In [ ]:
RNG = np.random.default_rng(20250816)
OUT = "."                      
MIN_TURNS = 2                  # scheme only applies to multi turn conversations
MIN_COVERAGE = 0.8             # drop conversations whose labeling did not cover the turns
N_BOOT = 5000

PLATFORM_FRAMES = {
    "claude": claude_trouble,
    "deepseek": deepseek_trouble,
    "chatgpt": chatgpt_trouble,
    "grok": grok_trouble,
    "gemini": gemini_trouble,
}
PLATFORMS = list(PLATFORM_FRAMES)


In [ ]:
# shared helpers 
from utils import boot_ci, cliffs_delta
def user_level(df, value_fn, name):
    """Apply value_fn to each user's rows, return one row per platform and user."""
    out = (
        df.groupby(["platform", "user_id"], sort=False)
        .apply(value_fn)
        .rename(name)
        .reset_index()
    )
    return out

def platform_summary(user_df, col):
    rows = []
    for p in PLATFORMS:
        v = user_df.loc[user_df["platform"] == p, col]
        m, lo, hi = boot_ci(v)
        rows.append(
            {
                "platform": p,
                "n_users": v.notna().sum(),
                "mean": m,
                "ci_lo": lo,
                "ci_hi": hi,
                "median": v.median(),
                "iqr_lo": v.quantile(0.25),
                "iqr_hi": v.quantile(0.75),
            }
        )
    return pd.DataFrame(rows)

def kruskal_across_platforms(user_df, col):
    groups = [user_df.loc[user_df["platform"] == p, col].dropna().values for p in PLATFORMS]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        return np.nan, np.nan, np.nan
    H, p = stats.kruskal(*groups)
    n = sum(len(g) for g in groups)
    k = len(groups)
    eps2 = (H - k + 1) / (n - k) if n > k else np.nan   # epsilon squared effect size
    return H, p, eps2

def pairwise_platforms(user_df, col):
    rows = []
    for i, p in enumerate(PLATFORMS):
        for q in PLATFORMS[i + 1:]:
            a = user_df.loc[user_df["platform"] == p, col]
            b = user_df.loc[user_df["platform"] == q, col]
            if a.dropna().size < 2 or b.dropna().size < 2:
                continue
            u, pv = stats.mannwhitneyu(a.dropna(), b.dropna(), alternative="two-sided")
            rows.append(
                {
                    "a": p, "b": q,
                    "mean_a": a.mean(), "mean_b": b.mean(),
                    "delta": cliffs_delta(a, b),
                    "p_raw": pv,
                }
            )
    out = pd.DataFrame(rows)
    if len(out):
        from statsmodels.stats.multitest import multipletests
        out["p_bh"] = multipletests(out["p_raw"], method="fdr_bh")[1]
    return out

In [ ]:
def _coerce(obj):
    """data_per_turn / conversation_label may arrive as str, list, dict or NaN."""
    if isinstance(obj, (list, dict)):
        return obj
    if obj is None or (isinstance(obj, float) and np.isnan(obj)):
        return None
    if isinstance(obj, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(obj)
            except Exception:
                continue
    return None


def explode_labels(df, platform):
    rows = []
    for rec in df.to_dict("records"):
        items = _coerce(rec.get("data_per_turn")) or []
        for it in items:
            if not isinstance(it, dict):
                continue
            rows.append(
                {
                    "platform": platform,
                    "conversation_id": rec["conversation_id"],
                    "user_id_lbl": rec.get("user_id"),
                    "turn_index": it.get("turn_index"),
                    "label": it.get("label"),
                    "attached": it.get("attached"),
                    "duplicate": it.get("duplicate"),
                    "position": it.get("position"),
                    "window": it.get("window"),
                    "conv_n_turns": rec.get("n_turns"),
                    "coverage": rec.get("coverage"),
                    "parse_error": rec.get("parse_error"),
                    "truncated": rec.get("truncated"),
                    "salvaged": rec.get("salvaged"),
                }
            )
    return pd.DataFrame(rows)


labels_long = pd.concat(
    [explode_labels(df, p) for p, df in PLATFORM_FRAMES.items()], ignore_index=True
)
labels_long["turn_index"] = pd.to_numeric(labels_long["turn_index"], errors="coerce")
labels_long = labels_long.dropna(subset=["turn_index"])
labels_long["turn_index"] = labels_long["turn_index"].astype(int)


In [ ]:
base = dataset.copy()
base["turn_index"] = pd.to_numeric(base["turn_index"], errors="coerce").astype("Int64")

turns = base.merge(
    labels_long.drop(columns=["platform", "user_id_lbl"]),
    on=["conversation_id", "turn_index"],
    how="left",
    validate="one_to_one",
)

# Inclusion: multi turn conversations, acceptable coverage, no parse failure, valid binary label
conv_len = turns.groupby("conversation_id")["turn_index"].transform("nunique")
keep = (
    conv_len.ge(MIN_TURNS)
    & turns["label"].isin(["trouble", "no_trouble"])
    & (pd.to_numeric(turns["coverage"], errors="coerce").fillna(0) >= MIN_COVERAGE)
    & (~turns["parse_error"].fillna(False).astype(bool))
)
turns = turns.loc[keep].copy()
turns["T"] = (turns["label"] == "trouble").astype(int)
turns = turns.sort_values(["platform", "user_id", "conversation_id", "turn_index"]).reset_index(drop=True)

# Position within conversation, needed for episode and termination logic
turns["pos"] = turns.groupby("conversation_id").cumcount()
turns["conv_len"] = turns.groupby("conversation_id")["turn_index"].transform("size")
turns["is_last"] = turns["pos"] == (turns["conv_len"] - 1)


# Trouble incidence

In [ ]:
user_conv_counts = turns.groupby(["platform", "user_id"])["conversation_id"].nunique()
eligible_users = user_conv_counts.reset_index()[["platform", "user_id"]]
turns = turns.merge(eligible_users, on=["platform", "user_id"], how="inner")

In [12]:
# Per user rate of trouble turns, then platform comparison over users.
# Also reported at the conversation level as the share of conversations containing any trouble.
# %%
inc_user = user_level(turns, lambda g: g["T"].mean(), "trouble_rate")
inc_user = inc_user.merge(
    user_level(turns, lambda g: g["conversation_id"].nunique(), "n_convs"),
    on=["platform", "user_id"],
)
inc_user = inc_user.merge(
    user_level(turns, lambda g: len(g), "n_turns"), on=["platform", "user_id"]
)

# conversation level: any trouble in the conversation, averaged within user first
conv_any = turns.groupby(["platform", "user_id", "conversation_id"])["T"].max().reset_index()
any_user = (
    conv_any.groupby(["platform", "user_id"])["T"].mean().rename("prop_convs_with_trouble").reset_index()
)
inc_user = inc_user.merge(any_user, on=["platform", "user_id"])

tbl_incidence = platform_summary(inc_user, "trouble_rate").assign(measure="trouble_rate_per_user")
tbl_any = platform_summary(inc_user, "prop_convs_with_trouble").assign(measure="prop_convs_with_trouble")
tbl_incidence = pd.concat([tbl_incidence, tbl_any], ignore_index=True)
tbl_incidence

,platform,n_users,mean,ci_lo,ci_hi,median,iqr_lo,iqr_hi,measure
0,claude,101,0.165282,0.141660,0.189874,0.151717,0.066667,0.244898,trouble_rate_per_user
1,deepseek,100,0.196311,0.170659,0.221792,0.168170,0.096803,0.277494,trouble_rate_per_user
2,chatgpt,100,0.118663,0.105505,0.132079,0.106943,0.069402,0.156366,trouble_rate_per_user
3,grok,98,0.163868,0.139662,0.191438,0.135893,0.064416,0.217033,trouble_rate_per_user
4,gemini,100,0.181966,0.156836,0.208621,0.161985,0.077730,0.248635,trouble_rate_per_user
5,claude,101,0.381227,0.336462,0.425660,0.393939,0.181818,0.542857,prop_convs_with_trouble
6,deepseek,100,0.357576,0.317212,0.398512,0.333333,0.218073,0.473434,prop_convs_with_trouble
7,chatgpt,100,0.278596,0.254098,0.303782,0.262801,0.200081,0.349560,prop_convs_with_trouble
8,grok,98,0.331457,0.292739,0.371456,0.334398,0.181818,0.437500,prop_convs_with_trouble
9,gemini,100,0.321537,0.284213,0.360028,0.306900,0.192047,0.441761,prop_convs_with_trouble


In [ ]:
# check stats wise the difference between the platforms
H, p, eps2 = kruskal_across_platforms(inc_user, "trouble_rate")
print(f"Kruskal Wallis on user trouble rate: H={H:.3f}, p={p:.4g}, eps2={eps2:.3f}")
pw_inc = pairwise_platforms(inc_user, "trouble_rate")
pw_inc

Kruskal Wallis on user trouble rate: H=19.499, p=0.000627, eps2=0.031


,a,b,mean_a,mean_b,delta,p_raw,p_bh
0,claude,deepseek,0.165282,0.196311,-0.139406,0.087942,0.146571
1,claude,chatgpt,0.165282,0.118663,0.191485,0.019076,0.063587
2,claude,grok,0.165282,0.163868,0.026066,0.751681,0.751681
3,claude,gemini,0.165282,0.181966,-0.061485,0.452127,0.502363
4,deepseek,chatgpt,0.196311,0.118663,0.355500,0.000014,0.000141
5,deepseek,grok,0.196311,0.163868,0.167041,0.042428,0.091405
6,deepseek,gemini,0.196311,0.181966,0.078100,0.340611,0.425764
7,chatgpt,grok,0.118663,0.163868,-0.164490,0.045703,0.091405
8,chatgpt,gemini,0.118663,0.181966,-0.262200,0.001364,0.006822
9,grok,gemini,0.163868,0.181966,-0.096020,0.243645,0.348064


## Linguistic characterization of trouble turns

In [ ]:
import re
import pandas as pd

# features we define for the linguistic caracterization are mainly from reading conversations with trouble 
# Then we create regex for detecting the features we notice
# Some features include shortend terms or slang because that is how users speak with the models 
# some features were tested but never used because they did not hold: exclamations, questions, elipses, etc. 

APOS = r"['\u2019\u02bc\u00b4`]?"

NEGATION = rf"""(?<!\w)(?:
    no|not|never|none|nothing|nobody|no\s?one|nowhere|nor|neither|
    cannot|can{APOS}t|couldn{APOS}t|
    don{APOS}t|doesn{APOS}t|didn{APOS}t|
    isn{APOS}t|aren{APOS}t|wasn{APOS}t|weren{APOS}t|ain{APOS}t|
    haven{APOS}t|hasn{APOS}t|hadn{APOS}t|
    won{APOS}t|wouldn{APOS}t|shan{APOS}t|shouldn{APOS}t|mustn{APOS}t|needn{APOS}t|
    dont|doesnt|didnt|cant|wont|isnt|arent|wasnt|werent|
    havent|hasnt|hadnt|couldnt|wouldnt|shouldnt
)(?!\w)"""

CORRECTION = rf"""(?<!\w)(?:
    actually|
    i\s+(?:meant|mean|said|asked|wanted|told\s+you)|
    that{APOS}?s\s+not|thats\s+not|
    (?:this|that|it)\s+is\s+not|
    not\s+what\s+i|
    instead\s+of|
    i\s+didn{APOS}?t\s+(?:ask|say|want|mean)|
    correction|
    (?:you|it)\s+(?:got|have|has)\s+(?:it|this|that)\s+wrong|
    (?:is|are|was|were)\s+(?:wrong|incorrect|inaccurate)|
    the\s+(?:correct|right)\s+\w+\s+is|
    should\s+(?:be|have\s+been)
)(?!\w)"""

REPEAT_MARKER = rf"""(?<!\w)(?:
    again|still|
    (?:once|yet)\s+more|
    as\s+i\s+(?:said|mentioned|already|told)|
    like\s+i\s+said|
    i\s+already\s+(?:said|told|asked|mentioned)|
    for\s+the\s+(?:second|third|last|nth)\s+time|
    keep\s+(?:doing|saying|giving|repeating)|
    same\s+(?:thing|answer|mistake|error)|
    you\s+(?:keep|always)\s+\w+ing
)(?!\w)"""

EVALUATIVE_NEG = r"""(?<!\w)(?:
    bad|terrible|awful|horrible|dreadful|atrocious|
    useless|worthless|pointless|meaningless|
    stupid|dumb|idiotic|moronic|
    nonsense|nonsensical|gibberish|garbage|trash|rubbish|junk|
    unhelpful|unusable|broken|
    lazy|sloppy|careless|
    disappointing|frustrating|annoying|infuriating|
    makes\s+no\s+sense|doesn.?t\s+make\s+sense|
    waste\s+of\s+(?:time|my\s+time)
)(?!\w)"""

EVALUATIVE_POS = r"""(?<!\w)(?:
    perfect|excellent|great|awesome|brilliant|wonderful|
    exactly|spot\s+on|nailed\s+it|
    much\s+better|way\s+better|
    helpful|useful|
    good\s+(?:job|work|answer|point)
)(?!\w)"""

POLITENESS = rf"""(?<!\w)(?:
    please|pls|plz|
    thank\s*(?:you|s)|thanks|thx|
    could\s+you|would\s+you|
    if\s+you\s+(?:could|would|don{APOS}?t\s+mind)|
    kindly|
    appreciate\s+it|
    i{APOS}?d\s+(?:like|prefer)|
    sorry|apolog(?:y|ies|ize)
)(?!\w)"""

HEDGE = rf"""(?<!\w)(?:
    maybe|perhaps|possibly|probably|
    might|may\s+be|could\s+be|
    i\s+(?:think|guess|suppose|believe|wonder)|
    sort\s+of|kind\s+of|kinda|sorta|
    somewhat|a\s+(?:bit|little)|
    seems?\s+(?:like|to)|
    not\s+(?:sure|certain)
)(?!\w)"""

SECOND_PERSON = rf"(?<!\w)(?:you|your|you{APOS}?re|youre|yours|yourself)(?!\w)"

ACCUSATORY = rf"""(?<!\w)(?:
    you\s+(?:didn{APOS}?t|don{APOS}?t|aren{APOS}?t|haven{APOS}?t|won{APOS}?t|
        can{APOS}?t|never|keep|always|still|failed|ignored|missed|
        misunderstood)|
    why\s+(?:did|do|are|would|can{APOS}?t|won{APOS}?t)\s+you|
    what\s+(?:are|were)\s+you\s+(?:doing|thinking)|
    your\s+(?:answer|response|reply|output|code)\s+(?:is|was)
)(?!\w)"""

PROFANITY = r"""(?<!\w)(?:
    f+u+c+k+\w*|f[\*\-_\.@#]+c?k\w*|fck\w*|fuk\w*|
    sh[i1\*]+t+\w*|sh[\*\-_\.@#]+t\w*|
    bull\s?sh[i1\*]+t|
    damn\w*|god\s?damn\w*|
    cr+a+p+\w*|
    a+s+s+hole|arsehole|
    wtf|stfu|ffs|omfg|
    piss(?:ed|ing)?\s+(?:off|me)|
    what\s+the\s+hell
)(?!\w)"""

IMPERATIVE_START = rf"""^[\s\W]*(?:(?:ok(?:ay)?|now|so|but|and|please|pls)\s+)?(?:
    give|show|write|make|do|stop|explain|list|fix|redo|retry|try|use|tell|
    answer|read|check|remove|add|change|update|rewrite|correct|revise|
    generate|create|provide|include|keep|put|send|find|search|look|
    forget|ignore|start|continue|don{APOS}?t
)(?!\w)"""

FEATURES = {
    "neg": NEGATION,
    "correction": CORRECTION,
    "repeat_marker": REPEAT_MARKER,
    "evaluative_neg": EVALUATIVE_NEG,
    "evaluative_pos": EVALUATIVE_POS,
    "politeness": POLITENESS,
    "hedge": HEDGE,
    "second_person": SECOND_PERSON,
    "accusatory": ACCUSATORY,
    "profanity": PROFANITY,
}

COMPILED = {name: re.compile(pat, re.IGNORECASE | re.VERBOSE)
            for name, pat in FEATURES.items()}
IMPERATIVE_RX = re.compile(IMPERATIVE_START, re.IGNORECASE | re.VERBOSE)
CODE_BLOCK_RX = re.compile(r"```.*?```|`[^`]+`", re.DOTALL)
CAPS_WORD_RX = re.compile(r"\b[A-Z]{2,}\b")
ALPHA_WORD_RX = re.compile(r"\b[A-Za-z]{2,}\b")
WORD_RX = re.compile(r"[a-z0-9']+")
ELLIPSIS_RX = re.compile(r"\.{3,}|\u2026")

CAPS_PROP_THRESHOLD = 0.9 # a message contains shouting if it at least 90% all caps (set after testing)
MIN_ALPHA_WORDS = 3 # the cutoff was 3 words --> it has to be 3 words in all caps before we say a message has shouting 


def featurize(text):
    t = str(text) if pd.notna(text) else ""
    prose = CODE_BLOCK_RX.sub(" ", t)
    words = WORD_RX.findall(t.lower())
    n = max(len(words), 1)

    f = {"n_words": len(words), "n_chars": len(t)}

    for name, rx in COMPILED.items():
        c = len(rx.findall(t))
        f[f"{name}_n"] = c
        f[f"{name}_rate"] = c / n
        f[f"has_{name}"] = int(c > 0)

    f["has_ellipsis"] = int(bool(ELLIPSIS_RX.search(t)))
    f["is_imperative"] = int(bool(IMPERATIVE_RX.search(t)))

    n_alpha = len(ALPHA_WORD_RX.findall(prose))
    caps_prop = len(CAPS_WORD_RX.findall(prose)) / n_alpha if n_alpha else 0.0
    f["caps_word_prop"] = caps_prop
    f["has_shouting"] = int(n_alpha >= MIN_ALPHA_WORDS
                            and caps_prop >= CAPS_PROP_THRESHOLD)
    return f


feat = pd.DataFrame([featurize(x) for x in turns["user_a"]], index=turns.index)
turns = pd.concat([turns.drop(columns=feat.columns, errors="ignore"), feat], axis=1)
turns["is_trouble"] = turns["label"].eq("trouble")

FEATURE_COLS = list(feat.columns)
print(len(FEATURE_COLS), "features |", turns["is_trouble"].sum(), "trouble turns")

36 features | 60074 trouble turns


In [15]:
lab = turns[turns["label"].notna()]
rate_cols = [c for c in FEATURE_COLS if c.endswith("_rate")]
has_cols = [c for c in FEATURE_COLS if c.startswith("has_")]

prev = pd.DataFrame({
    "pct_all": 100 * lab[has_cols].mean(),
    "pct_no_trouble": 100 * lab.loc[~lab["is_trouble"], has_cols].mean(),
    "pct_trouble": 100 * lab.loc[lab["is_trouble"], has_cols].mean(),
})
prev["ratio"] = prev["pct_trouble"] / prev["pct_no_trouble"].replace(0, np.nan)
display(prev.sort_values("ratio", ascending=False).round(2))

corr = lab[rate_cols].corr()
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
         .stack().rename("r").reset_index())
display(pairs[pairs["r"].abs() > 0.4].sort_values("r", ascending=False).round(3))

,pct_all,pct_no_trouble,pct_trouble,ratio
has_shouting,0.29,0.20,0.87,4.40
has_accusatory,1.38,1.08,3.16,2.94
has_profanity,1.67,1.32,3.72,2.82
has_correction,4.63,3.92,8.79,2.25
has_neg,23.26,19.77,43.72,2.21
has_repeat_marker,4.14,3.65,6.98,1.91
has_second_person,18.53,17.48,24.69,1.41
has_ellipsis,3.62,3.43,4.74,1.38
has_evaluative_neg,1.77,1.72,2.09,1.21
has_politeness,6.47,6.58,5.83,0.89


,level_0,level_1,r


In [18]:
lab = turns[turns["label"].notna()]
KEEP = ["neg", "correction", "repeat_marker", "accusatory",
        "profanity", "shouting", "politeness", "evaluative_pos", "second_person"]

rows = []
for f in KEEP:
    col = f"has_{f}"
    a = lab.loc[~lab["is_trouble"], col]
    b = lab.loc[lab["is_trouble"], col]
    tab = np.array([[b.sum(), len(b) - b.sum()], [a.sum(), len(a) - a.sum()]])
    chi2, p, _, _ = stats.chi2_contingency(tab)
    rows.append({"feature": f,
                 "pct_no_trouble": 100 * a.mean(),
                 "pct_trouble": 100 * b.mean(),
                 "ratio": b.mean() / a.mean() if a.mean() else np.nan,
                 "n_trouble_turns": int(b.sum()),
                 "p": p})

prof = pd.DataFrame(rows)
prof["p_bh"] = multipletests(prof["p"], method="fdr_bh")[1]
display(prof.sort_values("ratio", ascending=False).round(3))

print("\ntrouble turns with at least one marker:",
      f"{100 * lab.loc[lab['is_trouble'], [f'has_{f}' for f in KEEP[:6]]].max(axis=1).mean():.1f}%")

,feature,pct_no_trouble,pct_trouble,ratio,n_trouble_turns,p,p_bh
5,shouting,0.197,0.867,4.396,521,0.0,0.0
3,accusatory,1.075,3.158,2.937,1897,0.0,0.0
4,profanity,1.318,3.717,2.821,2233,0.0,0.0
1,correction,3.917,8.794,2.245,5283,0.0,0.0
0,neg,19.770,43.718,2.211,26263,0.0,0.0
2,repeat_marker,3.654,6.980,1.910,4193,0.0,0.0
8,second_person,17.482,24.695,1.413,14835,0.0,0.0
6,politeness,6.579,5.833,0.887,3504,0.0,0.0
7,evaluative_pos,2.395,1.465,0.612,880,0.0,0.0



trouble turns with at least one marker: 52.8%


## Recurrence and persistence

In [ ]:
# Transition probabilities computed within conversation, aggregated to the user, then to the platform.
def user_transitions(g):
    counts = {"N_to_N": 0, "N_to_T": 0, "T_to_N": 0, "T_to_T": 0}
    for _, conv in g.groupby("conversation_id"):
        s = conv.sort_values("turn_index")["T"].values
        for x, y in zip(s[:-1], s[1:]):
            key = ("T" if x else "N") + "_to_" + ("T" if y else "N")
            counts[key] += 1
    n_from_N = counts["N_to_N"] + counts["N_to_T"]
    n_from_T = counts["T_to_N"] + counts["T_to_T"]
    return pd.Series(
        {
            **counts,
            "p_T_given_N": counts["N_to_T"] / n_from_N if n_from_N else np.nan,
            "p_T_given_T": counts["T_to_T"] / n_from_T if n_from_T else np.nan,
            "n_from_N": n_from_N,
            "n_from_T": n_from_T,
        }
    )


trans_user = turns.groupby(["platform", "user_id"]).apply(user_transitions).reset_index()
trans_user["recurrence_ratio"] = trans_user["p_T_given_T"] / trans_user["p_T_given_N"]

tbl_trans = pd.concat(
    [
        platform_summary(trans_user, "p_T_given_N").assign(measure="p_T_given_N"),
        platform_summary(trans_user, "p_T_given_T").assign(measure="p_T_given_T"),
        platform_summary(trans_user.replace([np.inf, -np.inf], np.nan), "recurrence_ratio").assign(measure="recurrence_ratio"),
    ],
    ignore_index=True,
)
tbl_trans


,platform,n_users,mean,ci_lo,ci_hi,median,iqr_lo,iqr_hi,measure
0,claude,100,0.136877,0.114573,0.161363,0.105016,0.047619,0.196013,p_T_given_N
1,deepseek,99,0.162901,0.140013,0.186733,0.138889,0.069658,0.237664,p_T_given_N
2,chatgpt,100,0.097855,0.086598,0.109577,0.084449,0.060675,0.130018,p_T_given_N
3,grok,97,0.132367,0.108021,0.159930,0.108527,0.050420,0.197115,p_T_given_N
4,gemini,99,0.166378,0.141730,0.194341,0.132971,0.072234,0.227034,p_T_given_N
5,claude,87,0.393892,0.351071,0.438276,0.390244,0.256579,0.500000,p_T_given_T
6,deepseek,91,0.409051,0.357481,0.460323,0.428571,0.228459,0.572078,p_T_given_T
7,chatgpt,100,0.333262,0.309314,0.356315,0.359775,0.252641,0.404521,p_T_given_T
8,grok,87,0.427449,0.379889,0.473891,0.469388,0.300000,0.587768,p_T_given_T
9,gemini,94,0.403947,0.360639,0.447122,0.409185,0.291606,0.532284,p_T_given_T


In [ ]:
# %%
# Within user paired test: is trouble more likely after trouble than after no trouble
sub = trans_user[["p_T_given_N", "p_T_given_T"]].dropna()
print(stats.wilcoxon(sub["p_T_given_T"], sub["p_T_given_N"], zero_method="zsplit"))
for m in ["p_T_given_T", "recurrence_ratio"]:
    H, p, e = kruskal_across_platforms(trans_user.replace([np.inf, -np.inf], np.nan), m)
    print(f"{m}: H={H:.3f} p={p:.4g} eps2={e:.3f}")
pw_rec = pairwise_platforms(trans_user, "p_T_given_T")
pw_rec


WilcoxonResult(statistic=np.float64(3854.0), pvalue=np.float64(3.499162333349531e-66))
p_T_given_T: H=18.067 p=0.001197 eps2=0.031
recurrence_ratio: H=27.472 p=1.595e-05 eps2=0.053


,a,b,mean_a,mean_b,delta,p_raw,p_bh
0,claude,deepseek,0.393892,0.409051,-0.076797,0.376731,0.538188
1,claude,chatgpt,0.393892,0.333262,0.168161,0.047674,0.119186
2,claude,grok,0.393892,0.427449,-0.142555,0.104494,0.208989
3,claude,gemini,0.393892,0.403947,-0.067743,0.432125,0.540156
4,deepseek,chatgpt,0.409051,0.333262,0.268571,0.001363,0.005777
5,deepseek,grok,0.409051,0.427449,-0.052419,0.546339,0.607043
6,deepseek,gemini,0.409051,0.403947,0.026187,0.759115,0.759115
7,chatgpt,grok,0.333262,0.427449,-0.323103,0.000141,0.001409
8,chatgpt,gemini,0.333262,0.403947,-0.260532,0.001733,0.005777
9,grok,gemini,0.427449,0.403947,0.083884,0.330319,0.538188


## Analysis 5: trouble episodes and cascades

## Trajectory

In [ ]:
def episodes_for_seq(s):
    """Return list of episode lengths for a binary sequence."""
    eps, run = [], 0
    for v in s:
        if v == 1:
            run += 1
        elif run:
            eps.append(run)
            run = 0
    if run:
        eps.append(run)
    return eps


conv_rows = []
for (plat, uid, cid), conv in turns.groupby(["platform", "user_id", "conversation_id"], sort=False):
    s = conv.sort_values("turn_index")["T"].values
    eps = episodes_for_seq(s)
    conv_rows.append(
        {
            "platform": plat,
            "user_id": uid,
            "conversation_id": cid,
            "n_turns": len(s),
            "n_trouble": int(s.sum()),
            "trouble_rate": float(s.mean()),
            "n_episodes": len(eps),
            "mean_episode_len": np.mean(eps) if eps else np.nan,
            "max_episode_len": max(eps) if eps else 0,
            "has_trouble": int(s.sum() > 0),
            "has_cascade_3": int(any(e >= 3 for e in eps)),
            "ends_in_trouble": int(s[-1] == 1),
            "first_trouble_pos": int(np.argmax(s == 1)) if s.sum() else np.nan,
            "seq": "".join(map(str, s)),
        }
    )
convs = pd.DataFrame(conv_rows)
convs["rel_first_trouble"] = convs["first_trouble_pos"] / (convs["n_turns"] - 1).replace(0, np.nan)

In [ ]:
# Each conversation is reduced to a coarse trajectory type, then platforms are compared on the
# per user distribution of types.

# %%
def trajectory_type(row):
    s = row["seq"]
    n_t = s.count("1")
    if n_t == 0:
        return "no_trouble"
    eps = episodes_for_seq([int(c) for c in s])
    if max(eps) >= 3:
        return "prolonged"
    if len(eps) >= 2:
        return "recurrent"
    if s.endswith("1"):
        return "trouble_at_end"
    return "isolated"


convs["trajectory"] = convs.apply(trajectory_type, axis=1)

traj_user = (
    convs.groupby(["platform", "user_id", "trajectory"]).size().rename("n").reset_index()
)
traj_user["prop"] = traj_user["n"] / traj_user.groupby(["platform", "user_id"])["n"].transform("sum")
traj_tbl = (traj_user
            .pivot_table(index=["platform", "user_id"], columns="trajectory",
                         values="prop", aggfunc="sum", fill_value=0)
            .groupby("platform").mean())
traj_tbl


trajectory,isolated,no_trouble,prolonged,recurrent,trouble_at_end
platform,,,,,
chatgpt,0.092575,0.721404,0.039094,0.059838,0.087089
claude,0.103149,0.618773,0.080852,0.096340,0.100886
deepseek,0.106151,0.642424,0.060858,0.053314,0.137253
gemini,0.092705,0.678463,0.063763,0.053732,0.111336
grok,0.078953,0.668543,0.087139,0.059084,0.106281


In [ ]:
# Per type platform comparison over users
from statsmodels.stats.multitest import multipletests
traj_wide = traj_user.pivot_table(index=["platform", "user_id"], columns="trajectory",
                                  values="prop", aggfunc="sum", fill_value=0).reset_index()

rows = []
for t in ["no_trouble", "isolated", "recurrent", "prolonged", "trouble_at_end"]:
    groups = [g[t].values for _, g in traj_wide.groupby("platform")]
    H, p = stats.kruskal(*groups)
    eps2 = (H - len(groups) + 1) / (len(traj_wide) - len(groups))
    rows.append({"trajectory": t, "H": H, "p_raw": p, "eps2": eps2})

res = pd.DataFrame(rows)
res["p_bh"] = multipletests(res["p_raw"], method="fdr_bh")[1]
display(res.round(4))


,trajectory,H,p_raw,eps2,p_bh
0,no_trouble,13.8154,0.0079,0.0199,0.0132
1,isolated,5.8502,0.2106,0.0037,0.2633
2,recurrent,19.2144,0.0007,0.0308,0.0036
3,prolonged,0.8835,0.9269,-0.0063,0.9269
4,trouble_at_end,15.8817,0.0032,0.0241,0.0080
